# LSTM / Transformer — Percentage-Based Prediction Pipeline (v2)

**Predict a configurable fraction of the timeseries from a small initial reference
using SpatioTemporalLSTM or SpatioTemporalTransformer.**

### How it works
| Parameter | Meaning |
|-----------|--------|
| `INPUT_PCT` (k) | Percentage of timestep columns from the **beginning** used as reference input. |

* If `k = 10`, the first 10 % of the timesteps are used as reference and the remaining 90 % are predicted autoregressively.
* If `k = 30`, 30 % reference → 70 % predicted.

After inference the notebook plots **predictions vs ground truth** for a set of user-selected sample points.

### How to use
1. Set `INPUT_PCT`, `MODEL_TYPE`, data paths, and checkpoint in **Section 2**.
2. Run all cells in order.
3. Plots are displayed inline and saved to `OUTPUT_DIR`.

## 1 — Imports

In [ ]:
from __future__ import annotations

import math
import os
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2 — User Configuration

Set the **input percentage** (`INPUT_PCT`), model type, data paths, and checkpoint below.
Set `USE_MOCK = True` for a quick smoke test with synthetic data.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  USER SETTINGS — edit these before running                         ║
# ╚══════════════════════════════════════════════════════════════════════╝

INPUT_PCT = 30             # ← percentage of timesteps used as input (k)
                           #   e.g. 10 → first 10 % reference, predict 90 %
                           #        30 → first 30 % reference, predict 70 %

MODEL_TYPE = "SpatioTemporalLSTM"   # or "SpatioTemporalTransformer"

USE_MOCK = True            # ← set False when using real CSV data

CHECKPOINT_PATH = None     # e.g. "checkpoints/best_model.pth"
PRESSURE_CSV    = None     # e.g. "/path/to/master_pressure_2sec.csv"
U_VELOCITY_CSV  = None     # e.g. "/path/to/master_u_velocity_2sec.csv"
V_VELOCITY_CSV  = None     # e.g. "/path/to/master_v_velocity_2sec.csv"

OUTPUT_DIR      = "inference_outputs_lstm_transformer_v2"
N_SAMPLE_POINTS = 6        # ← number of sample points for time-series plots

# Architecture hyper-parameters (must match the trained checkpoint)
SPATIAL_EMBED_DIM = 64
HIDDEN_SIZE       = 256
NUM_LAYERS        = 3
NHEAD             = 8       # only used for Transformer
DROPOUT           = 0.2
K_NEIGHBORS       = 20
INPUT_SEQ_LENGTH  = 15      # lookback window the model was trained with
PRED_HORIZON      = 5       # single-step prediction horizon the model was trained with
BATCH_SIZE        = 256     # inference batch size

print(f"Model type : {MODEL_TYPE}")
print(f"Will use the first {INPUT_PCT}% of timesteps as input "
      f"and predict the remaining {100 - INPUT_PCT}%.")

## 3 — Data Loading

In [ ]:
def load_csv_data(
    pressure_path: str, u_vel_path: str, v_vel_path: str
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load three variable CSVs (x, y, t0, t1, …) → coords, pressure, u, v.

    Returns
    -------
    coords   : (N, 2)
    pressure : (N, T)
    u_vel    : (N, T)
    v_vel    : (N, T)
    """
    print("Loading CSVs …")
    df_p = pd.read_csv(pressure_path, header=0)
    df_u = pd.read_csv(u_vel_path, header=0)
    df_v = pd.read_csv(v_vel_path, header=0)

    assert len(df_p) == len(df_u) == len(df_v), (
        f"Row count mismatch: P={len(df_p)}, U={len(df_u)}, V={len(df_v)}"
    )
    coords = df_p.iloc[:, :2].values.astype(np.float32)
    pressure = df_p.iloc[:, 2:].values.astype(np.float32)
    u_vel = df_u.iloc[:, 2:].values.astype(np.float32)
    v_vel = df_v.iloc[:, 2:].values.astype(np.float32)

    T = min(pressure.shape[1], u_vel.shape[1], v_vel.shape[1])
    pressure, u_vel, v_vel = pressure[:, :T], u_vel[:, :T], v_vel[:, :T]
    print(f"  Loaded — nodes: {coords.shape[0]:,}, timesteps: {T}")
    return coords, pressure, u_vel, v_vel


def make_mock_data(
    n_nodes: int = 2_000, n_timesteps: int = 60, seed: int = 0
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Generate synthetic CFD-like data for testing."""
    rng = np.random.default_rng(seed)
    coords = rng.uniform(0, 1, (n_nodes, 2)).astype(np.float32)
    t = np.linspace(0, 2 * np.pi, n_timesteps, dtype=np.float32)
    x, y = coords[:, 0:1], coords[:, 1:2]
    pressure = (
        100_000
        + 5_000 * np.sin(2 * np.pi * x)
        + 3_000 * np.cos(2 * np.pi * y)
        + 1_000 * np.sin(t[np.newaxis, :])
        + rng.normal(0, 100, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)
    u_vel = (
        20
        + 10 * np.cos(np.pi * y)
        + 2 * np.sin(t[np.newaxis, :])
        + rng.normal(0, 0.5, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)
    v_vel = (
        5 * np.sin(np.pi * x)
        + 1 * np.cos(t[np.newaxis, :])
        + rng.normal(0, 0.3, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)
    print(f"  [Mock] Generated {n_nodes:,} nodes × {n_timesteps} timesteps.")
    return coords, pressure, u_vel, v_vel

print("Data loading functions ready.")

## 4 — Per-Variable Normalization

In [ ]:
def normalize_per_variable(
    data: np.ndarray, train_end_t: int
) -> Tuple[np.ndarray, List[StandardScaler]]:
    """
    Normalize each variable (P, U, V) independently using StandardScaler.
    Scalers are fit only on [0, train_end_t) to prevent data leakage.

    Parameters
    ----------
    data        : (n_cells, n_timesteps, 3)
    train_end_t : exclusive end index of the fitting range

    Returns
    -------
    data_normalized : (n_cells, n_timesteps, 3)
    scalers         : list of 3 StandardScaler
    """
    print("Normalizing per variable (fit on input split only) …")
    n_cells, n_timesteps, n_vars = data.shape
    data_normalized = np.empty_like(data)
    scalers: List[StandardScaler] = []
    var_names = ["Pressure", "U-velocity", "V-velocity"]

    for var_idx in range(n_vars):
        scaler = StandardScaler()
        scaler.fit(data[:, :train_end_t, var_idx].reshape(-1, 1))
        data_normalized[:, :, var_idx] = scaler.transform(
            data[:, :, var_idx].reshape(-1, 1)
        ).reshape(n_cells, n_timesteps)
        scalers.append(scaler)
        print(f"  {var_names[var_idx]}: mean={scaler.mean_[0]:.4f}, std={scaler.scale_[0]:.4f}")

    return data_normalized, scalers

print("Normalization ready.")

## 5 — Spatial Neighbourhood Graph

In [ ]:
def build_knn_graph(coords: np.ndarray, k: int) -> np.ndarray:
    """
    For each cell, find its k nearest spatial neighbours (excluding self).

    Returns
    -------
    neighbor_indices : (n_cells, k)
    """
    print(f"Building kNN graph with k={k} …")
    tree = cKDTree(coords)
    _, indices = tree.query(coords, k=k + 1)
    neighbor_indices = indices[:, 1:]
    print(f"  neighbor_indices shape: {neighbor_indices.shape}")
    return neighbor_indices

print("kNN graph builder ready.")

## 6 — Model Architecture

Both architectures must match the code used during training so that
checkpoint weights can be loaded correctly.

In [ ]:
class SpatialEncoder(nn.Module):
    """
    Encodes k-nearest-neighbour flow variables into a spatial embedding
    via a shared MLP followed by mean-pooling over the neighbour dimension.
    """

    def __init__(self, input_features: int = 3, embed_dim: int = 64) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_features, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, neighbor_seq: torch.Tensor) -> torch.Tensor:
        # neighbor_seq: (B, seq_len, k, 3)
        embedded = self.mlp(neighbor_seq)   # (B, seq_len, k, embed_dim)
        pooled = embedded.mean(dim=2)        # (B, seq_len, embed_dim)
        return pooled


class SpatioTemporalLSTM(nn.Module):
    """
    Spatiotemporal LSTM:
      1. SpatialEncoder aggregates neighbour features → spatial embedding
      2. Concatenate centre-cell features with spatial embedding
      3. LSTM processes the combined sequence
      4. Decoder MLP maps last hidden state → (pred_steps, 3)
    """

    def __init__(self, input_features: int = 3, spatial_embed_dim: int = 64,
                 hidden_size: int = 256, num_layers: int = 3,
                 pred_steps: int = 15, dropout: float = 0.2) -> None:
        super().__init__()
        self.pred_steps = pred_steps
        self.input_features = input_features
        self.spatial_encoder = SpatialEncoder(input_features, spatial_embed_dim)
        combined_dim = input_features + spatial_embed_dim
        self.lstm = nn.LSTM(
            combined_dim, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, pred_steps * input_features),
        )

    def forward(self, center_seq: torch.Tensor,
                neighbor_seq: torch.Tensor) -> torch.Tensor:
        spatial_embed = self.spatial_encoder(neighbor_seq)
        combined = torch.cat([center_seq, spatial_embed], dim=-1)
        lstm_out, _ = self.lstm(combined)
        last_hidden = lstm_out[:, -1, :]
        output = self.decoder(last_hidden)
        return output.view(-1, self.pred_steps, self.input_features)


class SpatioTemporalTransformer(nn.Module):
    """
    Spatiotemporal Transformer:
      1. SpatialEncoder aggregates neighbour features → spatial embedding
      2. Concatenate centre-cell features with spatial embedding
      3. Linear projection to d_model + learned positional encoding
      4. Transformer encoder processes the sequence
      5. Decoder MLP maps last-position output → (pred_steps, 3)
    """

    def __init__(self, input_features: int = 3, spatial_embed_dim: int = 64,
                 d_model: int = 256, nhead: int = 8, num_layers: int = 3,
                 pred_steps: int = 15, dropout: float = 0.2) -> None:
        super().__init__()
        self.pred_steps = pred_steps
        self.input_features = input_features
        self.spatial_encoder = SpatialEncoder(input_features, spatial_embed_dim)
        combined_dim = input_features + spatial_embed_dim
        self.input_projection = nn.Linear(combined_dim, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, 512, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4, dropout=dropout, batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps * input_features),
        )

    def forward(self, center_seq: torch.Tensor,
                neighbor_seq: torch.Tensor) -> torch.Tensor:
        spatial_embed = self.spatial_encoder(neighbor_seq)
        combined = torch.cat([center_seq, spatial_embed], dim=-1)
        x = self.input_projection(combined)
        seq_len = x.shape[1]
        x = x + self.pos_encoding[:, :seq_len, :]
        x = self.transformer_encoder(x)
        last_output = x[:, -1, :]
        output = self.decoder(last_output)
        return output.view(-1, self.pred_steps, self.input_features)


def create_model(model_type: str, pred_steps: int, spatial_embed_dim: int,
                 hidden_size: int, num_layers: int, nhead: int,
                 dropout: float) -> nn.Module:
    """Factory function — create model by name."""
    if model_type == "SpatioTemporalLSTM":
        model = SpatioTemporalLSTM(
            input_features=3, spatial_embed_dim=spatial_embed_dim,
            hidden_size=hidden_size, num_layers=num_layers,
            pred_steps=pred_steps, dropout=dropout,
        )
    elif model_type == "SpatioTemporalTransformer":
        model = SpatioTemporalTransformer(
            input_features=3, spatial_embed_dim=spatial_embed_dim,
            d_model=hidden_size, nhead=nhead, num_layers=num_layers,
            pred_steps=pred_steps, dropout=dropout,
        )
    else:
        raise ValueError(
            f"Unknown model_type: '{model_type}'. "
            "Choose 'SpatioTemporalLSTM' or 'SpatioTemporalTransformer'."
        )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Created {model_type} with {n_params:,} trainable parameters.")
    return model

print("Model definitions ready.")

## 7 — Load Trained Checkpoint

In [ ]:
def load_trained_model(
    model_type: str,
    checkpoint_path: Optional[str],
    pred_steps: int,
    spatial_embed_dim: int,
    hidden_size: int,
    num_layers: int,
    nhead: int,
    dropout: float,
    device: torch.device,
) -> nn.Module:
    """
    Create model architecture and optionally load trained weights.

    If *checkpoint_path* is None or does not exist, the model is returned with
    random weights (useful for testing the pipeline end-to-end).
    """
    model = create_model(
        model_type, pred_steps, spatial_embed_dim,
        hidden_size, num_layers, nhead, dropout,
    )
    if checkpoint_path and os.path.exists(checkpoint_path):
        state_dict = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(state_dict)
        print(f"✔ Loaded weights from {checkpoint_path}")
    else:
        if checkpoint_path:
            print(f"⚠ Checkpoint not found at {checkpoint_path}")
        print("  Using randomly initialised weights (for testing only).")
    model = model.to(device)
    model.eval()
    return model

print("Model loader ready.")

## 8 — Autoregressive Multi-Step Rollout

Given the first *k %* of the timeseries as reference, the model predicts
`pred_steps` timesteps at a time and feeds them back as input until the
remaining *(100 − k) %* timesteps have been generated.

In [ ]:
@torch.no_grad()
def autoregressive_rollout(
    model: nn.Module,
    data_normalized: np.ndarray,
    neighbor_indices: np.ndarray,
    input_end: int,
    seq_len: int,
    pred_steps: int,
    n_rollout_steps: int,
    batch_size: int,
    device: torch.device,
) -> np.ndarray:
    """
    Autoregressive multi-step prediction for all cells.

    The model uses the last *seq_len* timesteps of the history as input,
    predicts *pred_steps* future values, appends them to the history,
    and repeats until *n_rollout_steps* timesteps have been generated.

    Parameters
    ----------
    data_normalized  : (n_cells, n_timesteps, 3) — full normalised dataset.
    neighbor_indices : (n_cells, k) — kNN indices.
    input_end        : Index of the first timestep to predict (everything
                       before this is the reference / input window).
    seq_len          : Lookback window length the model expects.
    pred_steps       : Number of timesteps the model predicts per call.
    n_rollout_steps  : Total number of future timesteps to generate.
    batch_size       : Cells processed per forward pass.
    device           : Target device.

    Returns
    -------
    predictions : (n_cells, n_rollout_steps, 3)  — normalised
    """
    model.eval()
    n_cells = data_normalized.shape[0]

    # Start with the reference region as the initial history
    history = torch.FloatTensor(data_normalized[:, :input_end, :])
    nbr_tensor = torch.LongTensor(neighbor_indices)

    all_preds: List[np.ndarray] = []
    steps_generated = 0
    iteration = 0

    while steps_generated < n_rollout_steps:
        iteration += 1
        current_T = history.shape[1]
        batch_preds = []

        for start in range(0, n_cells, batch_size):
            end = min(start + batch_size, n_cells)
            center_seq = history[start:end, current_T - seq_len:, :]
            nbr_idx = nbr_tensor[start:end]
            neighbor_seq = history[nbr_idx, :, :]
            neighbor_seq = neighbor_seq[:, :, current_T - seq_len:, :].permute(0, 2, 1, 3)

            pred = model(center_seq.to(device), neighbor_seq.to(device))
            batch_preds.append(pred.cpu())

        new_steps = torch.cat(batch_preds, dim=0)  # (n_cells, pred_steps, 3)

        steps_to_take = min(pred_steps, n_rollout_steps - steps_generated)
        all_preds.append(new_steps[:, :steps_to_take, :].numpy())
        steps_generated += steps_to_take

        # Append to history for next iteration
        history = torch.cat([history, new_steps], dim=1)
        print(f"  Rollout iteration {iteration}: generated {steps_generated}/{n_rollout_steps} steps")

    return np.concatenate(all_preds, axis=1)  # (n_cells, n_rollout_steps, 3)

print("Autoregressive rollout ready.")

## 9 — Ground-Truth Extraction

In [ ]:
def extract_ground_truth(
    data: np.ndarray, input_end: int, n_steps: int
) -> np.ndarray:
    """
    Extract ground-truth values for the timesteps being predicted.

    Parameters
    ----------
    data      : (n_cells, n_timesteps, 3) — original-scale stacked array [P, U, V].
    input_end : First predicted timestep index.
    n_steps   : Number of timesteps to extract.

    Returns
    -------
    ground_truth : (n_cells, n_steps, 3)
    """
    return data[:, input_end : input_end + n_steps, :]

print("Ground-truth extraction ready.")

## 10 — Prediction vs Ground-Truth Plotting

For a set of sample nodes, plot the full timeseries showing:
- **Reference** (input) region as a solid line
- **Ground truth** in the prediction region as a dashed line
- **Model prediction** overlaid in a contrasting colour

In [ ]:
def plot_predictions_vs_ground_truth(
    data_physical: np.ndarray,
    predictions_physical: np.ndarray,
    input_end: int,
    coords: np.ndarray,
    sample_indices: np.ndarray,
    output_dir: str,
    input_pct: float,
    model_type: str,
) -> List[str]:
    """
    Plot predictions vs ground truth for selected sample points.

    For each variable (pressure, u_velocity, v_velocity), creates a figure
    with one subplot per sample node showing:
      • Reference input (first k % timesteps)
      • Ground truth (remaining timesteps)
      • Model prediction (overlaid on ground truth region)

    Parameters
    ----------
    data_physical          : (n_cells, n_timesteps, 3) — original-scale data.
    predictions_physical   : (n_cells, rollout_steps, 3) — predicted values
                             in physical units.
    input_end              : Timestep index where the reference window ends.
    coords                 : (N, 2) spatial coordinates.
    sample_indices         : (n_samples,) node indices to plot.
    output_dir             : Directory to save figures.
    input_pct              : Input percentage (for labelling).
    model_type             : Model name for the title.

    Returns
    -------
    paths : List of saved figure paths.
    """
    os.makedirs(output_dir, exist_ok=True)
    var_names = ["pressure", "u_velocity", "v_velocity"]
    T_total = data_physical.shape[1]
    rollout_steps = predictions_physical.shape[1]
    n_samples = len(sample_indices)
    paths: List[str] = []

    for v_idx, var_name in enumerate(var_names):
        n_cols = min(3, n_samples)
        n_rows = math.ceil(n_samples / n_cols)
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(6 * n_cols, 4 * n_rows),
            squeeze=False,
        )
        fig.suptitle(
            f"{model_type} — {var_name}\n"
            f"(input: first {input_pct:.0f}% = {input_end} timesteps, "
            f"predicted: {rollout_steps} timesteps)",
            fontsize=14, fontweight="bold",
        )

        for ax_idx, node_idx in enumerate(sample_indices):
            row, col = divmod(ax_idx, n_cols)
            ax = axes[row, col]

            gt_full = data_physical[node_idx, :, v_idx]
            t_all = np.arange(T_total)

            # Reference (input) region
            t_ref = t_all[:input_end]
            gt_ref = gt_full[:input_end]

            # Ground truth in prediction region
            t_pred = t_all[input_end : input_end + rollout_steps]
            gt_pred = gt_full[input_end : input_end + rollout_steps]

            # Model predictions
            pred_vals = predictions_physical[node_idx, :len(t_pred), v_idx]

            ax.plot(t_ref, gt_ref, color="steelblue", linewidth=1.2,
                    label="Reference (input)")
            ax.plot(t_pred, gt_pred, color="gray", linewidth=1.2,
                    linestyle="--", label="Ground truth")
            ax.plot(t_pred, pred_vals, color="crimson", linewidth=1.2,
                    label="Prediction")
            ax.axvline(x=input_end, color="black", linestyle=":",
                       linewidth=0.8, alpha=0.6)

            cx, cy = coords[node_idx]
            ax.set_title(f"Node {node_idx}  (x={cx:.3f}, y={cy:.3f})",
                         fontsize=10)
            ax.set_xlabel("Timestep")
            ax.set_ylabel(var_name)
            ax.legend(fontsize=8, loc="best")
            ax.grid(True, alpha=0.3)

        # Hide empty subplots
        for ax_idx in range(n_samples, n_rows * n_cols):
            row, col = divmod(ax_idx, n_cols)
            axes[row, col].set_visible(False)

        fig.tight_layout(rect=[0, 0, 1, 0.93])
        fig_path = os.path.join(output_dir, f"pred_vs_gt_{var_name}.png")
        fig.savefig(fig_path, dpi=150)
        plt.close(fig)
        paths.append(fig_path)
        print(f"  Saved: {fig_path}")

    return paths

print("Plotting function ready.")

## 11 — CSV Export

In [ ]:
def export_predictions_csv(
    coords: np.ndarray,
    predictions_norm: np.ndarray,
    scalers: List[StandardScaler],
    output_dir: str,
    prefix: str = "pred",
) -> List[str]:
    """
    Denormalise and write three CSV files (pressure, u_velocity, v_velocity).

    Parameters
    ----------
    coords           : (n_cells, 2)
    predictions_norm : (n_cells, n_steps, 3) — normalised predictions.
    scalers          : 3 fitted StandardScalers [P, U, V].
    output_dir       : Target directory.
    prefix           : Filename prefix.

    Returns
    -------
    paths : List of output CSV paths.
    """
    os.makedirs(output_dir, exist_ok=True)
    n_cells, n_steps, _ = predictions_norm.shape
    var_keys = ["pressure", "u_velocity", "v_velocity"]
    paths: List[str] = []

    for vi, key in enumerate(var_keys):
        pred_denorm = scalers[vi].inverse_transform(
            predictions_norm[:, :, vi].reshape(-1, 1)
        ).reshape(n_cells, n_steps)
        df = pd.DataFrame(coords, columns=["x_coord", "y_coord"])
        for t in range(n_steps):
            df[f"pred_t{t + 1}"] = pred_denorm[:, t]
        out_path = os.path.join(output_dir, f"{prefix}_{key}.csv")
        df.to_csv(out_path, index=False)
        paths.append(out_path)
        print(f"  Exported: {out_path}  ({n_cells:,} cells × {n_steps} steps)")

    return paths

print("CSV export ready.")

## 12 — Run Inference Pipeline

The pipeline:
1. Load data (real CSVs or synthetic mock)
2. Compute input / prediction split from `INPUT_PCT`
3. Build kNN spatial graph
4. Normalise data (fit on input portion only)
5. Create and optionally load the LSTM / Transformer model
6. Autoregressively predict the remaining timesteps
7. Compare with ground truth, export CSVs, and plot results

In [ ]:
def run_inference_v2(
    input_pct: float,
    model_type: str = "SpatioTemporalLSTM",
    checkpoint_path: Optional[str] = None,
    pressure_csv: Optional[str] = None,
    u_velocity_csv: Optional[str] = None,
    v_velocity_csv: Optional[str] = None,
    use_mock: bool = False,
    output_dir: str = "inference_outputs_lstm_transformer_v2",
    n_sample_points: int = 6,
    # architecture hyper-parameters
    spatial_embed_dim: int = 64,
    hidden_size: int = 256,
    num_layers: int = 3,
    nhead: int = 8,
    dropout: float = 0.2,
    k_neighbors: int = 20,
    input_seq_length: int = 15,
    pred_horizon: int = 5,
    batch_size: int = 256,
) -> Dict:
    """
    Percentage-based inference pipeline for SpatioTemporalLSTM / Transformer.

    Parameters
    ----------
    input_pct       : Percentage (0 < k < 100) of timesteps from the beginning
                      to use as reference input.  The remaining (100 − k) % are
                      predicted autoregressively.
    model_type      : "SpatioTemporalLSTM" or "SpatioTemporalTransformer".
    checkpoint_path : Path to a trained model checkpoint (.pth).
                      If None, a freshly initialised model is used (for testing).
    pressure_csv    : Path to the pressure CSV file.
    u_velocity_csv  : Path to the u-velocity CSV file.
    v_velocity_csv  : Path to the v-velocity CSV file.
    use_mock        : If True, generate synthetic data instead of loading CSVs.
    output_dir      : Directory where prediction CSVs and plots are written.
    n_sample_points : Number of sample nodes for time-series plots.

    Returns
    -------
    results : dict
    """
    if not (0 < input_pct < 100):
        raise ValueError(f"input_pct must be between 0 and 100 (exclusive), got {input_pct}")

    # ── 1. Load Data ──────────────────────────────────────────────────────────
    print("=" * 70)
    print(f"  {model_type} — Percentage-Based Inference Pipeline (v2)")
    print("=" * 70)
    print(f"  Input percentage     : {input_pct:.1f}%")
    print(f"  Model                : {model_type}")
    print(f"  Device               : {DEVICE}")
    print(f"  Checkpoint           : {checkpoint_path or '(none — random weights)'}")
    print(f"  Output directory     : {output_dir}")
    print("=" * 70)

    print("\n[1/7] Loading data …")
    if use_mock or not all(
        p and os.path.exists(p)
        for p in [pressure_csv, u_velocity_csv, v_velocity_csv]
    ):
        if not use_mock:
            print("  CSV files not found — falling back to mock data.")
        coords, pressure, u_vel, v_vel = make_mock_data()
    else:
        coords, pressure, u_vel, v_vel = load_csv_data(
            pressure_csv, u_velocity_csv, v_velocity_csv
        )

    N, T = pressure.shape
    print(f"  Nodes: {N:,}  |  Total timesteps: {T}")

    # ── 2. Compute input / prediction split ───────────────────────────────────
    print("\n[2/7] Computing input / prediction split …")
    input_end = max(int(math.ceil(T * input_pct / 100.0)), input_seq_length)
    if input_end >= T:
        raise ValueError(
            f"input_pct={input_pct}% gives {input_end} input timesteps "
            f"but only {T} total timesteps are available.  "
            f"Reduce input_pct so that at least 1 timestep remains for prediction."
        )
    n_steps = T - input_end

    print(f"  Total timesteps     : {T}")
    print(f"  Input  (reference)  : 0 … {input_end - 1}  ({input_end} timesteps = {input_end / T * 100:.1f}%)")
    print(f"  Predict             : {input_end} … {T - 1}  ({n_steps} timesteps = {n_steps / T * 100:.1f}%)")

    # ── 3. Build kNN graph ────────────────────────────────────────────────────
    print("\n[3/7] Building spatial neighbourhood graph …")
    neighbor_indices = build_knn_graph(coords, k_neighbors)

    # ── 4. Normalise ──────────────────────────────────────────────────────────
    print("\n[4/7] Normalising data …")
    # Stack raw data into (N, T, 3) for normalisation
    data_raw = np.stack([pressure, u_vel, v_vel], axis=-1)  # (N, T, 3)
    data_normalized, scalers = normalize_per_variable(data_raw, input_end)

    # ── 5. Build / load model ─────────────────────────────────────────────────
    print(f"\n[5/7] Building {model_type} …")
    model = load_trained_model(
        model_type, checkpoint_path, pred_horizon,
        spatial_embed_dim, hidden_size, num_layers, nhead, dropout, DEVICE,
    )

    # ── 6. Autoregressive rollout ─────────────────────────────────────────────
    print(f"\n[6/7] Running autoregressive rollout for {n_steps} timestep(s) …")
    rollout_norm = autoregressive_rollout(
        model, data_normalized, neighbor_indices,
        input_end=input_end,
        seq_len=input_seq_length,
        pred_steps=pred_horizon,
        n_rollout_steps=n_steps,
        batch_size=batch_size,
        device=DEVICE,
    )
    print(f"  ✔ Rollout complete — shape: {rollout_norm.shape}")

    # Denormalise predictions
    predictions_physical = np.empty_like(rollout_norm)
    for vi in range(3):
        predictions_physical[:, :, vi] = scalers[vi].inverse_transform(
            rollout_norm[:, :, vi].reshape(-1, 1)
        ).reshape(N, n_steps)

    # ── Ground truth ──────────────────────────────────────────────────────────
    ground_truth = extract_ground_truth(data_raw, input_end, n_steps)

    # ── Summary ───────────────────────────────────────────────────────────────
    var_names = ["pressure", "u_velocity", "v_velocity"]
    print("\n── Prediction Summary (first 5 nodes, denormalised) ──")
    for vi, vn in enumerate(var_names):
        print(f"\n  {vn}:")
        for node in range(min(5, N)):
            vals = " ".join(f"{predictions_physical[node, t, vi]:12.4f}"
                            for t in range(min(n_steps, 5)))
            print(f"    node {node:5d}: {vals}")

    # ── Export CSV ─────────────────────────────────────────────────────────────
    print(f"\n── Exporting predictions to {output_dir} ──")
    csv_paths = export_predictions_csv(
        coords, rollout_norm, scalers, output_dir,
        prefix=f"{model_type.lower()}_pred",
    )

    # ── 7. Plot predictions vs ground truth ───────────────────────────────────
    print(f"\n[7/7] Plotting predictions vs ground truth …")
    n_sample = min(n_sample_points, N)
    sample_indices = np.linspace(0, N - 1, n_sample, dtype=int)

    plot_paths = plot_predictions_vs_ground_truth(
        data_raw, predictions_physical,
        input_end, coords, sample_indices,
        output_dir, input_pct, model_type,
    )

    # Compute RMSE per variable
    print("\n── RMSE per variable (over all predicted timesteps & cells) ──")
    for vi, vn in enumerate(var_names):
        rmse = np.sqrt(np.mean(
            (predictions_physical[:, :, vi] - ground_truth[:, :, vi]) ** 2
        ))
        print(f"  {vn:15s}: RMSE = {rmse:.6f}")

    print("\n✔ Inference complete.")
    return {
        "predictions_norm": rollout_norm,
        "predictions_physical": predictions_physical,
        "ground_truth": ground_truth,
        "coords": coords,
        "scalers": scalers,
        "data_raw": data_raw,
        "data_normalized": data_normalized,
        "neighbor_indices": neighbor_indices,
        "config": {
            "model_type": model_type,
            "input_pct": input_pct,
            "input_end": input_end,
            "n_rollout_steps": n_steps,
        },
        "csv_paths": csv_paths,
        "plot_paths": plot_paths,
        "input_end": input_end,
        "sample_indices": sample_indices,
    }

## 13 — Execute

In [ ]:
results = run_inference_v2(
    input_pct=INPUT_PCT,
    model_type=MODEL_TYPE,
    checkpoint_path=CHECKPOINT_PATH,
    pressure_csv=PRESSURE_CSV,
    u_velocity_csv=U_VELOCITY_CSV,
    v_velocity_csv=V_VELOCITY_CSV,
    use_mock=USE_MOCK,
    output_dir=OUTPUT_DIR,
    n_sample_points=N_SAMPLE_POINTS,
    spatial_embed_dim=SPATIAL_EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    nhead=NHEAD,
    dropout=DROPOUT,
    k_neighbors=K_NEIGHBORS,
    input_seq_length=INPUT_SEQ_LENGTH,
    pred_horizon=PRED_HORIZON,
    batch_size=BATCH_SIZE,
)

## 14 — Inspect Results

In [ ]:
print("Prediction shape (normalised)  :", results["predictions_norm"].shape)
print("Prediction shape (physical)    :", results["predictions_physical"].shape)
print("Ground-truth shape             :", results["ground_truth"].shape)
print("Coordinate array shape         :", results["coords"].shape)
print(f"Input end timestep             : {results['input_end']}")
print(f"Sample node indices            : {results['sample_indices']}")
print(f"\nOutput CSVs:")
for p in results["csv_paths"]:
    print(f"  • {p}")
print("\nPlots:")
for p in results["plot_paths"]:
    print(f"  • {p}")